# GO Biological Process Ground Truth Collection

Build functional co-process ground truth from Gene Ontology **Biological Process** annotations, for the exp2 PPI-recovery benchmark.

GO-BP captures *shared biological process* — a functional/process axis complementary to physical complexes (CORUM, BioPlex), binary interaction (HuRI), and subcellular location (GO-CC). It overlaps somewhat with pathway sets (KEGG, Reactome), so it is best read as the canonical **functional reference** reviewers expect rather than a fully orthogonal evidence type.

**Key decisions (mirrors the GO-CC notebook):**
1. Aspect = `P` (Biological Process).
2. Evidence codes: experimental + curated, **exclude IEA** (bulk electronic).
3. **Generic-term filter** — this matters more for BP than any other GO aspect. Terms like *signal transduction* (~940 proteins), *regulation of transcription by RNA Pol II* (~1390) create huge near-meaningless cliques; two proteins both annotated to them is ~random. We drop terms annotating > `MAX_TERM_SIZE` proteins. BP's generic terms are larger/more numerous than CC's, so we use a tighter cutoff (**100**) than the GO-CC build (500).
4. Direct annotations only (no propagation up the ontology).

**Protein index:** built on the live **20431-protein** `ground_truth_proteins.tsv` in `protgpt/ppi_ground_truth/` (the layout every benchmark matrix shares), *not* the older FASTA-derived 20372 index the original GO-CC notebook used. Output is a drop-in column: `(20431, 20431)` matrix under key `matrix`, values `1` / `0` / `NaN`.

In [1]:
import gzip
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'protgpt' / 'ppi_ground_truth').is_dir() and (p / 'model').is_dir():
            return p
    raise FileNotFoundError('could not locate repo root from ' + str(start))

ROOT    = find_repo_root(Path.cwd())
GT_DIR  = ROOT / 'protgpt' / 'ppi_ground_truth'
GAF_CACHE = ROOT / 'data' / 'goa_human.gaf.gz'
OUT_NPZ = GT_DIR / 'gobp_ground_truth_matrix.npz'

MAX_TERM_SIZE = 100   # drop GO-BP terms annotating more than this many proteins (generic-term filter)

gt_proteins = pd.read_csv(GT_DIR / 'ground_truth_proteins.tsv', sep='\t').sort_values('idx')['protein'].to_numpy()
prot_to_idx = {p: i for i, p in enumerate(gt_proteins)}
n = len(gt_proteins)
print('protein index:', n, '(ground_truth_proteins.tsv)')

protein index: 20431 (ground_truth_proteins.tsv)


## 1. Download + parse the human GAF

GOA human annotation file. Cached locally so re-runs are offline. Columns used: db_object_id (col 2, UniProt acc), qualifier (4), go_id (5), evidence_code (7), aspect (9).

In [2]:
if GAF_CACHE.exists():
    raw = GAF_CACHE.read_bytes()
    print(f'GAF from cache: {len(raw)/1e6:.1f} MB')
else:
    url = 'http://current.geneontology.org/annotations/goa_human.gaf.gz'
    print('downloading', url)
    raw = requests.get(url, timeout=180).content
    GAF_CACHE.write_bytes(raw)
    print(f'downloaded + cached: {len(raw)/1e6:.1f} MB')

lines = gzip.decompress(raw).decode('utf-8').split('\n')
print(f'total lines: {len(lines):,}')

GAF from cache: 15.0 MB


total lines: 906,478

## 2. Filter to Biological Process, drop NOT, keep experimental + curated evidence

Exclude `IEA` (bulk electronic, noisy) — same evidence policy as the GO-CC build. Restrict to proteins present in the 20431 index.

In [3]:
experimental = {'EXP','IDA','IPI','IMP','IGI','IEP','HDA','HMP','HGI','HEP'}
curated      = {'TAS','NAS','IC','ISS','ISO','ISA','ISM','IBA','IBD','IKR','IRD','RCA'}
allowed = experimental | curated   # exclude IEA

term_to_prot = defaultdict(set)
n_ann = 0
for ln in lines:
    if not ln or ln[0] == '!':
        continue
    c = ln.split('\t')
    if len(c) < 15 or c[8] != 'P':         # aspect P = Biological Process
        continue
    if 'NOT' in c[3] or c[6] not in allowed:
        continue
    if c[1] in prot_to_idx:                 # restrict to the 20431 index
        term_to_prot[c[4]].add(c[1])
        n_ann += 1
term_to_prot = dict(term_to_prot)
print(f'GO-BP annotations (exp+curated, in index): {n_ann:,}')
print(f'unique GO-BP terms: {len(term_to_prot):,}')

GO-BP annotations (exp+curated, in index): 130,412
unique GO-BP terms: 10,136


## 3. Generic-term filter

The largest BP terms are exactly the uninformative ones (transcription regulation, signal transduction). Sweep cutoffs, then drop terms above `MAX_TERM_SIZE`.

In [4]:
sizes = {t: len(p) for t, p in term_to_prot.items()}
print('largest (generic) GO-BP terms:')
for t, s in sorted(sizes.items(), key=lambda kv: -kv[1])[:10]:
    print(f'  {t}: {s:>5} proteins')

print(f"\n{'cutoff':>7} {'terms kept':>11} {'proteins':>9}")
for c in [50, 100, 200, 500, 1000]:
    kept = {t: p for t, p in term_to_prot.items() if len(p) <= c}
    cov = len(set().union(*kept.values())) if kept else 0
    print(f'{c:>7} {len(kept):>11} {cov:>9}')

term_to_prot_f = {t: p for t, p in term_to_prot.items() if len(p) <= MAX_TERM_SIZE}
print(f'\nMAX_TERM_SIZE = {MAX_TERM_SIZE}: kept {len(term_to_prot_f)} terms '
      f'covering {len(set().union(*term_to_prot_f.values())):,} proteins')

largest (generic) GO-BP terms:
  GO:0006357:  1390 proteins
  GO:0045944:   975 proteins
  GO:0007165:   937 proteins
  GO:0000122:   803 proteins
  GO:0045893:   559 proteins
  GO:0006355:   500 proteins
  GO:0045892:   481 proteins
  GO:0007186:   466 proteins
  GO:0006955:   433 proteins
  GO:0008284:   393 proteins

 cutoff  terms kept  proteins
     50        9848     13321
    100       10031     14390
    200       10099     15263
    500       10131     16137
   1000       10135     16365

MAX_TERM_SIZE = 100: kept 10031 terms covering 14,390 proteins


## 4. Build the (20431×20431) ground-truth matrix

`1` = share ≥ 1 (filtered) BP term; `0` = both annotated but share none; `NaN` = at least one protein unannotated. Same structure/semantics as the other ground-truth matrices.

In [5]:
prot_terms = defaultdict(set)
for t, ps in term_to_prot_f.items():
    for p in ps:
        prot_terms[p].add(t)
ann_proteins = sorted(prot_terms)
all_terms = sorted({t for ts in prot_terms.values() for t in ts})
t_idx = {t: i for i, t in enumerate(all_terms)}

membership = np.zeros((len(ann_proteins), len(all_terms)), dtype=np.float32)
for i, p in enumerate(ann_proteins):
    for t in prot_terms[p]:
        membership[i, t_idx[t]] = 1.0
shared = membership @ membership.T

full_idx = np.array([prot_to_idx[p] for p in ann_proteins])
gobp_matrix = np.full((n, n), np.nan, dtype=np.float32)
gobp_matrix[np.ix_(full_idx, full_idx)] = 0.0                 # both annotated -> negative
pi, pj = np.where(shared >= 1)
m = pi != pj
gobp_matrix[full_idx[pi[m]], full_idx[pj[m]]] = 1.0            # share >=1 term -> positive
np.fill_diagonal(gobp_matrix, np.nan)

up = gobp_matrix[np.triu_indices(n, 1)]
n_pos = int(np.nansum(up == 1)); n_neg = int(np.nansum(up == 0)); n_nan = int(np.isnan(up).sum())
print(f'annotated proteins: {len(ann_proteins):,}')
print(f'upper triangle  positives: {n_pos:,}  negatives: {n_neg:,}  NaN: {n_nan:,}')
print(f'pos/neg ratio: {n_pos/n_neg:.4f}')
print(f'symmetric: {np.allclose(gobp_matrix, gobp_matrix.T, equal_nan=True)}')

annotated proteins: 14,390
upper triangle  positives: 812,068  negatives: 102,716,787  NaN: 105,173,810
pos/neg ratio: 0.0079


symmetric: True


## 5. Save (drop-in column for `protgpt/ppi_ground_truth/`)

In [6]:
from pathlib import Path
from omicsfm.attention import GT_DIR_DEFAULT, save_ppi_ground_truth

# save_ppi_ground_truth stores only the positives and the covered-protein
# universe (~36x smaller, reconstructs the dense matrix exactly) and will
# not replace an existing file: published results are pinned to the
# matrices in reference/, and these source databases change over time.
save_ppi_ground_truth(Path(GT_DIR_DEFAULT) / "gobp_ground_truth_matrix.npz", gobp_matrix)
print(f'saved -> {OUT_NPZ}  ({OUT_NPZ.stat().st_size/1e6:.1f} MB)')
print('parameters: aspect=P | evidence=experimental+curated (no IEA) | '
      f'max_term_size={MAX_TERM_SIZE} | direct annotations | threshold >=1 shared term')
print('shares ground_truth_proteins.tsv with every other db -> loads as db="gobp".')

saved -> C:\Users\sander\OneDrive\Bureaublad\Projects\prot_GPT\code\protgpt\protgpt\ppi_ground_truth\gobp_ground_truth_matrix.npz  (63.7 MB)
parameters: aspect=P | evidence=experimental+curated (no IEA) | max_term_size=100 | direct annotations | threshold >=1 shared term
shares ground_truth_proteins.tsv with every other db -> loads as db="gobp".
